# TURNA - Turkce Metin Siniflandirma Kiyaslamasi

**Katman:** Turkce transformer

Turkce icin uretilmis encoder-decoder - siniflandirma + aciklama uretimi. (Not: seq2seq; burada encoder ozellik cikarici olarak siniflandiriliyor.)

Gorev: Turkce deprem ihbar metni -> 5 ust kategori (+ opsiyonel 1-5 aciliyet).  
Veri: `data/` (bkz `data/README.md`).  
Butun defterler `common.py` uzerinden AYNI bolme ve AYNI metrikleri kullanir.  
Metrik: Dogruluk, makro-F1, sinif bazli P/R, konfuzyon matrisi, **yanlis negatif orani**.

In [ ]:
# === COLAB KURULUM ===  (yerelde calistirinca atlanir)
import os, sys
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB and not os.path.exists('/content/QuakeMind'):
    !git clone --depth 1 https://github.com/UtBird/QuakeMind.git /content/QuakeMind
if IN_COLAB:
    %cd /content/QuakeMind/model_benchmarks/04_turkce_metin
    %pip install -q transformers datasets accelerate evaluate
print('cwd:', os.getcwd())

## 1. Veri

In [ ]:
import sys, warnings; warnings.filterwarnings('ignore')
from pathlib import Path
sys.path.insert(0, str(Path.cwd()))
import numpy as np, pandas as pd
import common as C

train_df, val_df, test_df = C.load_text_dataset()
labels, l2i, i2l = C.label_maps(train_df)
print('siniflar:', labels)
print('train/val/test:', len(train_df), len(val_df), len(test_df))
y_test = test_df['label'].map(l2i).values

## 2. Model ve tokenizer

In [ ]:
MODEL_NAME = "turna"
HF_MODEL = "boun-tabi-LMG/TURNA"
PARAMS = dict(hf_model=HF_MODEL, epochs=4, lr=2e-5, batch_size=16, max_len=128)

import torch
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          TrainingArguments, Trainer, DataCollatorWithPadding)
from datasets import Dataset

tok = AutoTokenizer.from_pretrained(HF_MODEL)

def to_ds(df):
    d = Dataset.from_pandas(df[['text', 'label']].assign(label=df['label'].map(l2i)))
    return d.map(lambda b: tok(b['text'], truncation=True, max_length=PARAMS['max_len']), batched=True)

ds_tr, ds_va, ds_te = to_ds(train_df), to_ds(val_df), to_ds(test_df)
model = AutoModelForSequenceClassification.from_pretrained(
    HF_MODEL, num_labels=len(labels),
    id2label=i2l, label2id=l2i)

## 3. Egitim

In [ ]:
import evaluate; f1_metric = evaluate.load('f1')
def compute_metrics(p):
    preds = np.argmax(p.predictions, axis=1)
    return f1_metric.compute(predictions=preds, references=p.label_ids, average='macro')

args = TrainingArguments(
    output_dir=f'./_hf_{MODEL_NAME}', num_train_epochs=PARAMS['epochs'],
    learning_rate=PARAMS['lr'], per_device_train_batch_size=PARAMS['batch_size'],
    per_device_eval_batch_size=32, eval_strategy='epoch', save_strategy='no',
    logging_steps=50, report_to='none', seed=C.SEED)

trainer = Trainer(model=model, args=args, train_dataset=ds_tr, eval_dataset=ds_va,
                  tokenizer=tok, data_collator=DataCollatorWithPadding(tok),
                  compute_metrics=compute_metrics)
trainer.train()

## 4. Test tahmini + sonuc

In [ ]:
y_pred = np.argmax(trainer.predict(ds_te).predictions, axis=1)

In [ ]:
m = C.classification_metrics(y_test, y_pred, labels)
print('accuracy :', round(m['accuracy'], 4))
print('macro-F1 :', round(m['macro_f1'], 4))
print('macro yanlis-negatif orani:', round(m['macro_fnr'], 4))
print()
print(m['report'])
lb = C.save_result(MODEL_NAME, m, extra={'params': PARAMS})
lb

In [ ]:
import matplotlib.pyplot as plt
cm = np.array(m['confusion_matrix'])
fig, ax = plt.subplots(figsize=(5, 4))
im = ax.imshow(cm, cmap='Blues')
ax.set_xticks(range(len(labels))); ax.set_xticklabels(labels, rotation=45, ha='right')
ax.set_yticks(range(len(labels))); ax.set_yticklabels(labels)
for (i, j), v in np.ndenumerate(cm): ax.text(j, i, int(v), ha='center', va='center')
ax.set_xlabel('tahmin'); ax.set_ylabel('gercek'); ax.set_title(f'{MODEL_NAME} konfuzyon')
plt.tight_layout(); plt.savefig(f'results/{MODEL_NAME}_cm.png', dpi=140); plt.show()